In [ ]:
import wandb
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize the API
api = wandb.Api(api_key="6da3cfca616fa8b7e812fc8fcf54ef2a08870da2")

# Get your run (replace with your entity/project/run_id)

runs = api.runs("kirdon6-university-of-copenhagen/structure-prediction")
#[print(r.name) for r in runs[:10]]
pretrain_runs = [r for r in runs if r.name == "mlm-transformer-800k"]


# Create subplots - 2x2 grid
sns.set_theme(style="whitegrid", palette="viridis", font_scale=1.1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Training Metrics Over Time', fontsize=20, fontweight='bold', y=0.98)

# Define the metrics and their properties
metrics = [
    {'column': 'train/loss', 'title': 'Training Loss', 'ylabel': 'Loss', 'ylim_top': 1.4, 'ylim_bottom': None},
    {'column': 'train/accuracy', 'title': 'Training Accuracy', 'ylabel': 'Accuracy', 'ylim_top': None, 'ylim_bottom': 0.3},
    {'column': 'val/loss', 'title': 'Validation Loss', 'ylabel': 'Loss', 'ylim_top': 1.4, 'ylim_bottom': None},
    {'column': 'val/accuracy', 'title': 'Validation Accuracy', 'ylabel': 'Accuracy', 'ylim_top': None, 'ylim_bottom': 0.3}
]

# Store handles and labels for the legend
handles, labels = [], []

# Plot each metric
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]  # Get the appropriate subplot
    
    for run in pretrain_runs:
        history = run.history()
        
        # Extract the specific metric
        data = history[['_step', metric['column']]]
        tag = run.tags[0]
        
        # Plot the metric
        line = ax.plot(data['_step'], data[metric['column']], label=tag)
        
        # Collect handles and labels for legend (only from first subplot to avoid duplicates)
        if idx == 0:
            handles.extend(line)
            labels.append(tag)
    
    # Customize each subplot
    ax.set_xlabel('Epochs', fontweight='bold')
    ax.set_ylabel(metric['ylabel'], fontweight='bold')
    ax.set_title(metric['title'], fontsize=12, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set y-limits if specified
    if metric['ylim_top']:
        ax.set_ylim(top=metric['ylim_top'])
    if metric['ylim_bottom']:
        ax.set_ylim(bottom=metric['ylim_bottom'])

# Add single legend at the bottom with 3 columns
fig.legend(handles, labels, 
          loc='lower center', 
          bbox_to_anchor=(0.5, -0.02),
          ncol=3,
          frameon=True,
          fancybox=True,
          shadow=True,
          fontsize=10)

plt.tight_layout()
plt.subplots_adjust(top=0.93, bottom=0.1)  # Make room for title and legend

plt.show()
